# DART 공시 수집 대상 기업 목록 만들기

`dart_collector.ipynb`가 읽는 `source/targets.csv`를 만든다. 기준은 `graphDB.md` 2장 "공시 수집 대상"을 따른다.

| 순서 | 묶음 | 기준 |
|---|---|---|
| 1 | 코스피200 | `market_data/kospi200*.csv`의 구성 종목 |
| 2 | 나머지 코스피 | 코스피 상장 기업 전체에서 사업 회사가 아닌 곳(리츠, 인프라펀드, 해외 기업)을 뺀 것 |
| 3 | 코스닥150 | `market_data/kosdaq150*.csv`의 구성 종목 |
| 4 | 추가 코스닥 | 아래 세 조건을 모두 만족하는 코스닥 종목 중 코스닥150이 아닌 것 |

추가 코스닥 조건 (최근 20거래일 KRX 코스닥 일별매매정보 기준)

- 소속부가 우량기업부·중견기업부·기술성장기업부·벤처기업부 (관리종목, 투자주의환기종목, SPAC, 외국기업 제외)
- 위 소속부 종목 중 20거래일 평균 시가총액 상위 300개 안
- 20거래일 평균 거래대금 10억원 이상

`targets.csv`는 위 순서대로 저장한다. `dart_collector`는 이 순서대로 요청하므로, 하루 한도에 걸려 멈춰도 앞쪽(중요한) 기업부터 받아 둔 상태가 된다.

**입력 파일**

- `market_data/sector.json`: 상장 기업 전체와 시장 구분, WI26 업종
- `market_data/kospi200*.csv`, `market_data/kosdaq150*.csv`: KRX 정보데이터시스템에서 받은 지수 구성종목 (EUC-KR). 파일이 여러 개면 이름순으로 마지막 것을 쓴다(파일 이름에 날짜를 넣어 두면 최신 파일이 쓰인다).
- 프로젝트 루트 `.env`의 `KRX_API_KEY`: KRX Open API 인증키

KRX 일별매매정보는 날짜마다 `source/krx/` 아래에 저장해 두고, 이미 받은 날짜는 다시 요청하지 않는다.

**한눈에**

| | |
|---|---|
| 처리 순서에서 | **`dart_targets`** (처음) → `dart_collector` |
| 외부 요청·비용 | 있음. KRX Open API(무료) 일별매매정보 최근 20거래일 × 2시장. 받은 날짜는 `source/krx/`에 저장해 다시 요청하지 않는다 |
| 다시 실행 | 된다. 새 날짜만 받는다 |
| 결과 | `source/targets.csv`(1,067개, 코스피200 → 코스피 → 코스닥150 → 추가 코스닥 순서) |

## 1. 설정

In [1]:
# ── 설정 ──
import datetime as dt
import json
import re
import time
from pathlib import Path

import pandas as pd
import requests

# 이 노트북은 graph_db 폴더에서 실행한다.
ROOT = Path.cwd()
assert ROOT.name == 'graph_db', 'graph_db 폴더에서 실행해야 한다.'

MARKET_DATA = ROOT.parent / 'market_data'      # 여러 파트가 같이 쓰는 시장 원자료
KRX_DIR = ROOT / 'source' / 'krx'              # KRX 일별매매정보 원본 (날짜별 JSON)
OUT_PATH = ROOT / 'source' / 'targets.csv'     # 결과: dart_collector가 읽는 대상 기업 목록
KRX_DIR.mkdir(parents=True, exist_ok=True)

# ── 추가 코스닥 기준 (graphDB.md 2장) ─────────────────────────────────────────
N_DAYS = 20                         # 평균을 낼 거래일 수
TOP_N_BY_MKTCAP = 300               # 평균 시가총액 순위 기준
MIN_TRADING_VALUE = 1_000_000_000   # 평균 거래대금 하한 (10억원)
OK_SECTIONS = {'우량기업부', '중견기업부', '기술성장기업부', '벤처기업부'}   # 통과시키는 코스닥 소속부

# ── 요청량 추정 (dart_collector 기준) ─────────────────────────────────────────
DART_REQUESTS_PER_COMPANY = 17      # 기업개황 1 + 정기보고서 4 + 수시공시 10 + 검색·원문 2
DART_DAILY_LIMIT = 20_000

In [2]:
# ── 인증키: .env에서 읽는다(노트북에 쓰지 않는다) ──
def load_env_value(name):
    """graph_db 또는 프로젝트 루트의 .env에서 값을 읽는다."""
    for env_path in [ROOT / '.env', ROOT.parent / '.env']:
        if not env_path.exists():
            continue
        for line in env_path.read_text(encoding='utf-8').splitlines():
            if line.startswith(name + '='):
                return line.split('=', 1)[1].strip().strip('"\'')
    raise RuntimeError(f'.env에서 {name}를 찾지 못했다.')

KRX_API_KEY = load_env_value('KRX_API_KEY')
print('KRX 키 확인:', len(KRX_API_KEY), '자리')

KRX 키 확인: 40 자리


## 2. 상장 기업 전체와 지수 구성종목

- `sector.json`의 종목 전체(KONEX, 우선주는 이미 빠져 있다)가 출발점이다.
- 코스닥 글로벌(`KOSDAQ GLOBAL`)은 코스닥 안의 우량 기업 세그먼트이므로 코스닥으로 본다.

In [3]:
# ── 2. 상장 기업 전체(sector.json)와 지수 구성종목 ──
sector = json.loads((MARKET_DATA / 'sector.json').read_text(encoding='utf-8'))
universe = pd.DataFrame([
    {'stock_code': s['security_code'],
     'name': s['security_name'],
     'market': 'KOSPI' if s['market'] == 'KOSPI' else 'KOSDAQ',   # KOSDAQ GLOBAL → KOSDAQ
     'kosdaq_global': s['market'] == 'KOSDAQ GLOBAL',
     'wi26_sector': s['wi26']['sector_name']}
    for s in sector['stocks'].values()
])
print('상장 기업', len(universe), '개:', universe['market'].value_counts().to_dict(),
      '/ 코스닥 글로벌', int(universe['kosdaq_global'].sum()), '개')


def load_index_members(pattern):
    """market_data에서 지수 구성종목 CSV(KRX 정보데이터시스템, EUC-KR)를 읽어 종목코드 집합을 돌려준다."""
    files = sorted(MARKET_DATA.glob(pattern))
    if not files:
        raise FileNotFoundError(f'market_data/{pattern} 파일이 없다.')
    path = files[-1]                      # 여러 개면 이름순 마지막 (날짜를 붙여 두면 최신)
    df = pd.read_csv(path, dtype={'종목코드': str}, encoding='euc-kr')
    codes = set(df['종목코드'].str.zfill(6))
    print(f'{path.name}: {len(codes)}개')
    return codes

KOSPI200 = load_index_members('kospi200*.csv')
KOSDAQ150 = load_index_members('kosdaq150*.csv')

# 지수 구성종목이 sector.json에 없으면 목록에서 빠지므로 확인한다
for label, codes in [('코스피200', KOSPI200), ('코스닥150', KOSDAQ150)]:
    missing = codes - set(universe['stock_code'])
    if missing:
        print(f'  ! {label} 중 sector.json에 없는 종목:', sorted(missing))

universe['kospi200'] = universe['stock_code'].isin(KOSPI200)
universe['kosdaq150'] = universe['stock_code'].isin(KOSDAQ150)

상장 기업 2655 개: {'KOSDAQ': 1826, 'KOSPI': 829} / 코스닥 글로벌 50 개
kospi200.csv: 201개
kosdaq150.csv: 150개


## 3. KRX 일별매매정보 받기 (최근 20거래일)

KRX Open API의 유가증권·코스닥 일별매매정보를 날짜별로 받는다. 한 번 요청에 그날 전 종목이 온다.

- 어제부터 하루씩 거슬러 올라가며 받는다. 오늘 자료는 장이 끝난 뒤에야 확정되므로 쓰지 않는다.
- 주말은 요청하지 않는다. 공휴일은 빈 응답이 오므로 거래일로 세지 않는다.
- 받은 날짜는 `source/krx/시장_날짜.json`에 저장하고, 다음 실행에서는 파일을 읽는다.

In [4]:
# ── 3. KRX 일별매매정보: 날짜별로 받고 저장 ──
KRX_BASE = 'https://data-dbg.krx.co.kr/svc/apis/sto/'
KRX_ENDPOINTS = {'KOSPI': 'stk_bydd_trd', 'KOSDAQ': 'ksq_bydd_trd'}


def krx_daily(market, day):
    """하루치 일별매매정보(전 종목)를 돌려준다. 휴장일이면 빈 리스트.

    저장된 파일이 있으면 읽고, 없으면 요청해서 저장한다.
    """
    path = KRX_DIR / f'{market}_{day:%Y%m%d}.json'
    if path.exists():
        return json.loads(path.read_text(encoding='utf-8'))
    response = requests.get(KRX_BASE + KRX_ENDPOINTS[market], headers={'AUTH_KEY': KRX_API_KEY},
                            params={'basDd': f'{day:%Y%m%d}'}, timeout=60)
    time.sleep(0.5)
    if response.status_code != 200:
        raise RuntimeError(f'KRX {market} {day} 요청 실패: {response.status_code} {response.text[:200]}')
    rows = response.json().get('OutBlock_1', [])
    path.write_text(json.dumps(rows, ensure_ascii=False), encoding='utf-8')
    return rows


def recent_trading_days(n_days, max_lookback=45):
    """어제부터 거슬러 올라가며 자료가 있는 거래일 n_days개를 찾는다 (코스닥 기준)."""
    days, day = [], dt.date.today() - dt.timedelta(days=1)
    for _ in range(max_lookback):
        if day.weekday() < 5 and krx_daily('KOSDAQ', day):      # 평일이고 자료가 있으면 거래일
            days.append(day)
            if len(days) == n_days:
                return days
        day -= dt.timedelta(days=1)
    raise RuntimeError(f'최근 {max_lookback}일 안에서 거래일 {n_days}개를 찾지 못했다.')


DAYS = recent_trading_days(N_DAYS)
print(f'거래일 {len(DAYS)}개: {DAYS[-1]} ~ {DAYS[0]}')

frames = []
for market in KRX_ENDPOINTS:
    for day in DAYS:
        frames.append(pd.DataFrame(krx_daily(market, day)).assign(market=market))
daily = pd.concat(frames, ignore_index=True)
for col in ['MKTCAP', 'ACC_TRDVAL']:
    daily[col] = pd.to_numeric(daily[col].str.replace(',', ''), errors='coerce')
print('일별매매정보', len(daily), '행')

거래일 20개: 2026-10-13 ~ 2026-11-09
일별매매정보 55294 행


In [5]:
# ── 3. 종목별 20거래일 평균 시가총액·거래대금, 최근 소속부 ──
# 종목별 20거래일 평균. 소속부는 가장 최근 날짜의 값을 쓴다.
latest_section = (daily.sort_values('BAS_DD')
                       .groupby('ISU_CD')['SECT_TP_NM'].last())
stats = (daily.groupby('ISU_CD')
              .agg(mktcap_avg=('MKTCAP', 'mean'), trdval_avg=('ACC_TRDVAL', 'mean'), days=('BAS_DD', 'nunique'))
              .join(latest_section.rename('section'))
              .reset_index()
              .rename(columns={'ISU_CD': 'stock_code'}))

# 억원 단위로 보기 쉽게
stats['mktcap_avg_억'] = (stats['mktcap_avg'] / 1e8).round(0)
stats['trdval_avg_억'] = (stats['trdval_avg'] / 1e8).round(1)
universe = universe.merge(stats[['stock_code', 'section', 'mktcap_avg_억', 'trdval_avg_억', 'days']],
                          on='stock_code', how='left')

# 코스닥 글로벌 종목이 KRX에서 어떤 소속부로 나오는지 확인 (소속부 기준에서 잘못 빠지지 않는지)
print('코스닥 글로벌 종목의 소속부:',
      universe.loc[universe['kosdaq_global'], 'section'].value_counts(dropna=False).to_dict())

코스닥 글로벌 종목의 소속부: {'우량기업부': 39, '중견기업부': 8, '기술성장기업부': 3}


## 4. 코스피: 사업 회사가 아닌 곳 빼기

KRX 종목기본정보 API(증권구분)는 지금 인증키로 쓸 권한이 없어서 이름과 종목코드로 거른다.

| 규칙 | 걸리는 것 | 주의 |
|---|---|---|
| 이름이 "리츠"로 끝남 | 부동산투자회사 (SK리츠, 롯데리츠 등) | "리츠"가 들어가기만 하면 메리츠금융지주가 잘못 걸리므로 **끝나는 것만** 본다 |
| 이름이 "인프라"로 끝남 | 인프라펀드 (맥쿼리인프라, KB발해인프라) | |
| 종목코드가 9로 시작 | 해외 기업 (외국 주권, DR) | |
| 직접 지정 | 이름 규칙에 안 걸리는 리츠 | 이리츠코크렙, 맵스리얼티 |

코스닥은 5절의 KRX 소속부(SPAC, 외국기업 등)로 거르므로 이 규칙을 쓰지 않는다.

In [6]:
# ── 4. 코스피: 리츠·인프라펀드·해외 기업 같은 사업 회사가 아닌 곳 빼기 ──
# 이름 규칙에 걸리지 않는 리츠를 직접 적는다 (2026-11-10 sector.json 코스피 종목을 확인해서 찾음)
NON_BUSINESS_BY_CODE = {
    '088260': '리츠',      # 이리츠코크렙
    '094800': '리츠',      # 맵스리얼티
}


def non_business_reason(stock_code, name):
    """사업 회사가 아니면 그 이유를, 사업 회사면 빈 문자열을 돌려준다."""
    if stock_code in NON_BUSINESS_BY_CODE:
        return NON_BUSINESS_BY_CODE[stock_code]
    if name.endswith('리츠'):
        return '리츠'
    if name.endswith('인프라'):
        return '인프라펀드'
    if stock_code.startswith('9'):
        return '해외 기업'
    return ''


kospi = universe[universe['market'] == 'KOSPI'].copy()
kospi['exclude_reason'] = [non_business_reason(c, n) for c, n in zip(kospi['stock_code'], kospi['name'])]
kospi_excluded = kospi[kospi['exclude_reason'] != '']
kospi = kospi[kospi['exclude_reason'] == ''].drop(columns='exclude_reason')

print('코스피', len(kospi) + len(kospi_excluded), '개 중', len(kospi_excluded), '개 제외 →', len(kospi), '개')
if kospi_excluded['kospi200'].any():
    print('  ! 코스피200 종목이 제외되었다. 규칙을 확인한다:', kospi_excluded.loc[kospi_excluded['kospi200'], 'name'].tolist())
kospi_excluded[['stock_code', 'name', 'exclude_reason']]

코스피 829 개 중 28 개 제외 → 801 개


,stock_code,name,exclude_reason
163,0030R0,대신밸류리츠,리츠
1327,088260,이리츠코크렙,리츠
1336,088980,맥쿼리인프라,인프라펀드
1398,094800,맵스리얼티,리츠
1661,145270,케이탑리츠,리츠
2096,293940,신한알파리츠,리츠
2202,330590,롯데리츠,리츠
2215,334890,이지스밸류플러스리츠,리츠
2225,338100,NH프라임리츠,리츠
2253,348950,제이알글로벌리츠,리츠


## 5. 코스닥: 추가할 종목 고르기

1. 소속부가 `OK_SECTIONS`인 종목만 남긴다. 코스닥 글로벌은 소속부와 관계없이 통과시킨다.
   - 해외 기업(종목코드가 9로 시작)은 소속부가 있어도 뺀다. 예: 950260 인제니아테라퓨틱스(Reg.S)는 KRX에서 '외국기업'이 아니라 '기술성장기업부'로 나온다.
2. 남은 종목을 20거래일 평균 시가총액 순으로 세워 상위 300개를 남긴다.
3. 20거래일 평균 거래대금이 10억원 미만이면 뺀다.
4. 코스닥150에 이미 있는 종목은 3번 묶음에 들어가므로 여기서 뺀다.

코스닥150은 지수 구성종목이므로 위 조건과 관계없이 모두 넣는다.

In [7]:
# ── 5. 코스닥: 소속부 → 시가총액 상위 300 → 거래대금 10억원 이상, 코스닥150이 아닌 것 ──
kosdaq = universe[universe['market'] == 'KOSDAQ'].copy()

# 1) 소속부. 해외 기업은 KRX 소속부가 '외국기업'이 아닌 경우가 있어 종목코드로 한 번 더 거른다
kosdaq['foreign'] = kosdaq['stock_code'].str.startswith('9')
kosdaq['section_ok'] = (kosdaq['section'].isin(OK_SECTIONS) | kosdaq['kosdaq_global']) & ~kosdaq['foreign']
# 2) 소속부를 통과한 종목 안에서의 시가총액 순위
kosdaq['mktcap_rank'] = kosdaq['mktcap_avg_억'].where(kosdaq['section_ok']).rank(ascending=False, method='first')
kosdaq['top_mktcap'] = kosdaq['mktcap_rank'] <= TOP_N_BY_MKTCAP
# 3) 거래대금
kosdaq['liquid'] = kosdaq['trdval_avg_억'] * 1e8 >= MIN_TRADING_VALUE

kosdaq['meets_criteria'] = kosdaq['section_ok'] & kosdaq['top_mktcap'] & kosdaq['liquid']
kosdaq_extra = kosdaq[kosdaq['meets_criteria'] & ~kosdaq['kosdaq150']]

print('코스닥', len(kosdaq), '개')
print('  소속부 통과', int(kosdaq['section_ok'].sum()), '→ 시총 상위', int(kosdaq['top_mktcap'].sum()),
      '→ 거래대금 통과', int(kosdaq['meets_criteria'].sum()))
print('  그중 코스닥150', int((kosdaq['meets_criteria'] & kosdaq['kosdaq150']).sum()),
      '/ 추가', len(kosdaq_extra))

# 코스닥150인데 기준에는 못 미치는 종목 (지수 종목이라 그대로 넣는다. 참고용)
kosdaq[kosdaq['kosdaq150'] & ~kosdaq['meets_criteria']][
    ['stock_code', 'name', 'section', 'mktcap_rank', 'mktcap_avg_억', 'trdval_avg_억']]

코스닥 1826 개
  소속부 통과 1570 → 시총 상위 300 → 거래대금 통과 258
  그중 코스닥150 142 / 추가 116


,stock_code,name,section,mktcap_rank,mktcap_avg_억,trdval_avg_억
525,015750,성우하이텍,우량기업부,178.0,4285.0,8.6
667,025900,동화기업,중견기업부,253.0,3050.0,3.8
713,032190,다우데이타,우량기업부,123.0,6446.0,9.0
801,036620,감성코퍼레이션,우량기업부,242.0,3189.0,8.8
1466,101730,위메이드맥스,중견기업부,237.0,3329.0,6.7
1769,194480,데브시스터즈,중견기업부,494.0,1474.0,6.8
1826,211050,인카금융서비스,우량기업부,163.0,4762.0,6.0
2259,352480,씨앤씨인터내셔널,우량기업부,275.0,2666.0,3.4


## 6. 목록 저장

네 묶음을 순서대로 붙여 저장한다. 묶음 안에서는 평균 시가총액이 큰 순서다.

In [8]:
# ── 6. 네 묶음을 순서대로 붙여 저장 ──
GROUPS = [
    (1, '코스피200', kospi[kospi['kospi200']]),
    (2, '코스피', kospi[~kospi['kospi200']]),
    (3, '코스닥150', kosdaq[kosdaq['kosdaq150']]),
    (4, '코스닥 추가', kosdaq_extra),
]
targets = pd.concat([
    df.assign(priority=p, group=label).sort_values('mktcap_avg_억', ascending=False)
    for p, label, df in GROUPS
], ignore_index=True)

COLUMNS = ['stock_code', 'name', 'market', 'priority', 'group', 'wi26_sector', 'kospi200', 'kosdaq150',
           'kosdaq_global', 'section', 'mktcap_avg_억', 'trdval_avg_억']
targets = targets[COLUMNS]
assert targets['stock_code'].is_unique, '같은 종목이 두 묶음에 들어갔다.'

targets.to_csv(OUT_PATH, index=False, encoding='utf-8-sig')    # 엑셀에서 열어도 한글이 깨지지 않게 BOM을 붙인다
print('저장:', OUT_PATH.relative_to(ROOT), len(targets), '개')
print(f'KRX 자료 기간: {DAYS[-1]} ~ {DAYS[0]}')

summary = targets.groupby(['priority', 'group']).size().rename('기업 수').reset_index()
summary['DART 요청(추정)'] = summary['기업 수'] * DART_REQUESTS_PER_COMPANY
summary['누적 요청'] = summary['DART 요청(추정)'].cumsum()
summary['하루 한도 안'] = summary['누적 요청'] <= DART_DAILY_LIMIT
summary

저장: source/targets.csv 1067 개
KRX 자료 기간: 2026-10-13 ~ 2026-11-09


,priority,group,기업 수,DART 요청(추정),누적 요청,하루 한도 안
0,1,코스피200,201,3417,3417,True
1,2,코스피,600,10200,13617,True
2,3,코스닥150,150,2550,16167,True
3,4,코스닥 추가,116,1972,18139,True


In [9]:
# ── 확인: 묶음별 WI26 업종 분포 ──
# 업종 분포: 묶음별로 어느 WI26 업종이 들어갔는지
targets.pivot_table(index='wi26_sector', columns='group', values='stock_code', aggfunc='count', fill_value=0) \
       [[g for _, g, _ in GROUPS]].sort_values('코스피200', ascending=False)

group,코스피200,코스피,코스닥150,코스닥 추가
wi26_sector,,,,
화학,21,46,4,2
"상사,자본재",15,22,3,3
건강관리,14,49,27,25
필수소비재,13,41,1,0
"화장품,의류",12,55,6,1
기계,11,24,12,11
자동차,11,63,3,3
소프트웨어,11,12,17,8
은행,10,7,2,2
